# 1. Extraction des données (lxml)

Ce notebook extrait les paragraphes des comptes rendus de l’Assemblée nationale à partir des fichiers XML (en utilisant la bibliothèque lxml) et retourne un csv exploitable dans la suite de l'analyse.

In [ ]:
# TODO : check nb extract contre fichiers nosdeputés (en virant les italiques etc.)
# - TODO: déduplication ? -> proposé un truc déjà, vérif (pas sur soit utile)
#   - plus tard : # choisir la clé la plus pertinente
#   - (["uid", "id_syceron", "texte"] VS uid + id_syceron seulement)
#   - en réalité encore des choses qui ont double entrée pour même ID_paragraphe
#   - mais avec texte différent = des didascalies, texte italique, etc.
#   - si pas de Texte, 370 lignes supprimées (mais qui vireraient sans doute au cleaning des données)
# - TODO: vérifier si on a pas d'autres doublons de fichier mal placés dans les législatures
#   - check si possible automatiser à la lecture de tous les uid vs seance ref, etc. ?
# - TODO: plus tard, aviser récupération des points de contexte parents(cf tentative Matthias)



In [ ]:
# # TODO : ON PEUT CHOPER LA DUPLICATION DE CAS PAR L'ID_SYCERON ET TEXTES !!
# # TODO : tests congrès et pb séances doublons.

# mask_congres = df["session"].str.contains("Congrès du Parlement", case=False, na=False)

# print("Nombre de lignes concernées :", int(mask_congres.sum()))

# # Optionnel : voir les valeurs exactes
# df.loc[mask_congres, "session"].value_counts()

# 1.1 définition fonctions d'extraction

In [ ]:
import os
import glob
from lxml import etree
import pandas as pd

# ==================================================================
# FONCTIONS D'EXTRACTION DES DONNÉES
# ==================================================================


# ======== Fonction extraction infos depuis fichier XML =========
def extraire_paragraphes_lxml(fichier_xml: str) -> pd.DataFrame:
    """
    Extrait les paragraphes d'un fichier XML de compte rendu en utilisant lxml.
    """
    try:
        tree = etree.parse(fichier_xml)
        root = tree.getroot()
        ns = {"ns": "http://schemas.assemblee-nationale.fr/referentiel"}

        meta = {
            "uid": root.findtext("ns:uid", namespaces=ns),
            "SeanceRef": root.findtext("ns:seanceRef", namespaces=ns),  # pas partout
            "SessionRef": root.findtext("ns:sessionRef", namespaces=ns),  # pas partout
        }
        meta_tags = [
            "dateSeance",
            "dateSeanceJour",
            "numSeanceJour",
            "numSeance",
            "typeAssemblee",
            "legislature",
            "session",
            "nomFichierJo",
            "presidentSeance",
        ]
        for tag in meta_tags:
            meta[tag] = root.findtext(f".//ns:{tag}", namespaces=ns)

        rows = []

        for paragraphe in root.xpath(".//ns:paragraphe", namespaces=ns):
            # Naviguer vers le <point> parent
            # récupérer les infos
            point = paragraphe.getparent()
            while point is not None and point.tag != f"{{{ns['ns']}}}point":
                point = point.getparent()

            point_type = point.get("code_grammaire") if point is not None else None

            # anciennement : utilisait findtext(), mais ignore les sous-balises donc perte de texte
            # plutôt utiliser itertext() pour reconstruire le contenu complet
            texte_point = (
                point.find("ns:texte", namespaces=ns) if point is not None else None
            )
            point_title = (
                "".join(texte_point.itertext()).strip()
                if texte_point is not None
                else None
            )

            # # Plus pris pour l'instant (ie niveau du point, on a toujours niveau paragraphe plus bas):
            # point_id = point.get("id_syceron") if point is not None else None
            # point_valeur_ptsodj = point.get("valeur_ptsodj") if point is not None else None

            texte_elem = paragraphe.find("ns:texte", namespaces=ns)
            texte = (
                "".join(texte_elem.itertext()).strip()
                if texte_elem is not None
                else None
            )
            stime = texte_elem.get("stime") if texte_elem is not None else None

            # Récupérer les informations de l'orateur
            # ie celles présentes dans la balise <orateur>
            # et pas forcément dans les attributs du paragraphe
            orateur = paragraphe.find(".//ns:orateur", namespaces=ns)
            nom_orateur = (
                orateur.findtext("ns:nom", namespaces=ns)
                if orateur is not None
                else None
            )
            qualite_orateur = (
                orateur.findtext("ns:qualite", namespaces=ns)
                if orateur is not None
                else None
            )
            id_orateur = (
                orateur.findtext("ns:id", namespaces=ns)
                if orateur is not None
                else None
            )

            # toper désormais toutes les infos
            # garder apparent pour éventuels choix ou recodages des noms plutôt que des machins type `**meta`
            rows.append(
                {
                    # ========================
                    # Métadonnées de la séance
                    # ========================
                    "uid": meta["uid"],
                    "SeanceRef": meta["SeanceRef"],
                    "SessionRef": meta["SessionRef"],
                    "dateSeance": meta["dateSeance"],
                    "dateSeanceJour": meta["dateSeanceJour"],
                    "numSeanceJour": meta["numSeanceJour"],
                    "numSeance": meta["numSeance"],
                    "typeAssemblee": meta["typeAssemblee"],
                    "legislature": meta["legislature"],
                    "session": meta["session"],
                    "nomFichierJo": meta["nomFichierJo"],
                    "presidentSeance": meta["presidentSeance"],
                    # ========================
                    # Données du point parent (contexte)
                    # ========================
                    "point_titre": point_title,
                    "point_type": point_type,
                    # 'Sous_titre': '',  # not in this version, get back to original if needed
                    # 'Contexte_hierarchique': '',  # not in this version, get back to original if needed
                    # 'Section_courante': '',  # not in this version, get back to original if needed
                    # 'Sujet_point': '', # not in this version, get back to original if needed
                    # "point_valeur_ptsodj": point_valeur_ptsodj,
                    # "point_id": point_id,
                    # ========================
                    # données du paragraphe
                    # ========================
                    "valeur_ptsodj": paragraphe.get("valeur_ptsodj"),
                    "ordinal_prise": paragraphe.get("ordinal_prise"),
                    "ordre_absolu_seance": paragraphe.get("ordre_absolu_seance"),
                    "id_acteur": paragraphe.get("id_acteur"),
                    "id_mandat": paragraphe.get("id_mandat"),
                    "code_grammaire": paragraphe.get("code_grammaire"),
                    "code_style": paragraphe.get("code_style"),
                    "code_parole": paragraphe.get("code_parole"),
                    "id_syceron": paragraphe.get("id_syceron"),
                    "roledebat": paragraphe.get("roledebat"),
                    # ========================
                    # données orateur + texte
                    # ========================
                    "nom_orateur": nom_orateur,
                    "qualite_orateur": qualite_orateur,
                    "id_orateur": id_orateur,
                    "stime": stime,
                    "texte": texte,
                }
            )

        return pd.DataFrame(rows)

    except Exception as e:
        print(f" Erreur dans {fichier_xml} : {e}")
        return pd.DataFrame()


# ======== Fonction traitement d'un dossier contenant les XML =========
def traiter_dossier_compte_rendu_lxml(
    dossier_path: str, pattern: str = "*.xml"
) -> pd.DataFrame:
    """
    Traite tous les fichiers XML d'un dossier avec la fonction extraire_paragraphes_lxml().
    """
    fichiers = glob.glob(os.path.join(dossier_path, pattern))
    if not fichiers:
        print(f"Aucun fichier XML trouvé dans {dossier_path}")
        return pd.DataFrame()

    all_dfs = []
    total = len(fichiers)
    print(f"Traitement de {total} fichiers XML...\n")

    for i, fichier in enumerate(fichiers, 1):
        nom = os.path.basename(fichier)
        print(f"[{i}/{total}] {nom}...", end=" ")

        df = extraire_paragraphes_lxml(fichier)
        if not df.empty:
            print(f"{len(df)} lignes")
            all_dfs.append(df)
        else:
            print("Vide ou erreur")

    if all_dfs:
        df_final = pd.concat(all_dfs, ignore_index=True)
        print(f"\n Export terminé : {len(df_final)} lignes consolidées")
        return df_final
    else:
        return pd.DataFrame()


In [1]:
## 1.2 Extraction des données des XML et export CSV

In [ ]:
# ==================================================================
# TRAITEMENT DES LÉGISLATURES SOUHAITÉES
# ==================================================================

# ==========Traitement de la 16° législature==========
df_16 = traiter_dossier_compte_rendu_lxml("../data/raw/16-xml/compteRendu/")

# Pour le sous cas de la 16 législature : nettoyer le fichier qui n'est pas au bon endroit
# ie "faux" fichier "CRSANR5L16S2021O1N144" qui date de 2021
# et est bien présent dans la 15 sous le nom CRSANR5L15S2021O1N144
df_16 = df_16[df_16["uid"] != "CRSANR5L16S2021O1N144"]
print("suppression de CRSANR5L16S2021O1N144")

# export
df_16.to_csv("../data/interim/extract_16.csv", index=False, encoding="utf-8")
print(f"\n Export CSV : ({df_16.shape[0]} lignes)")

# ==========Traitement de la 15° législature==========
df_15 = traiter_dossier_compte_rendu_lxml("../data/raw/15-xml/compteRendu/")
df_15.to_csv("../data/interim/extract_15.csv", index=False, encoding="utf-8")
print(f"\n Export CSV : ({df_15.shape[0]} lignes)")

Traitement de 605 fichiers XML...

[1/605] CRSANR5L16S2023O1N173.xml... 633 lignes
[2/605] CRSANR5L16S2023O1N167.xml... 467 lignes
[3/605] CRSANR5L16S2023O1N198.xml... 402 lignes
[4/605] CRSANR5L16S2023E1N010.xml... 818 lignes
[5/605] CRSANR5L16S2024O1N117.xml... 678 lignes
[6/605] CRSANR5L16S2024O1N103.xml... 857 lignes
[7/605] CRSANR5L16S2023E1N004.xml... 646 lignes
[8/605] CRSANR5L16S2023O1N239.xml... 452 lignes
[9/605] CRSANR5L16S2023O1N205.xml... 118 lignes
[10/605] CRSANR5L16S2023O1N211.xml... 296 lignes
[11/605] CRSANR5L16S2024O1N088.xml... 403 lignes
[12/605] CRSANR5L16S2023O1N007.xml... 535 lignes
[13/605] CRSANR5L16S2023O1N013.xml... 808 lignes
[14/605] CRSANR5L16S2024O1N063.xml... 106 lignes
[15/605] CRSANR5L16S2024O1N077.xml... 760 lignes
[16/605] CRSANR5L16S2024O1N076.xml... 834 lignes
[17/605] CRSANR5L16S2024O1N062.xml... 322 lignes
[18/605] CRSANR5L16S2023O1N012.xml... 688 lignes
[19/605] CRSANR5L16S2023O1N006.xml... 430 lignes
[20/605] CRSANR5L16S2024O1N089.xml... 541 l

## 1.3 Fusion des législatures

In [ ]:
# ==================================================================
# FUSION DES LÉGISLATURES
# concaténation de df_15 et df_16 (ou lecture depuis CSV si nécessaire)
# ==================================================================

# # si déjà en mémoire : utiliser df_15, df_16 ; sinon :
# df_15 = pd.read_csv("../data/interim/extract_15.csv", encoding="utf-8")
# df_16 = pd.read_csv("../data/interim/extract_16.csv", encoding="utf-8")

# si besoin de vérifier et aligner les colonnes
# Mais overkill ici, on est propre normalement

# cols15 = set(df_15.columns)
# cols16 = set(df_16.columns)
# for c in sorted((cols15 | cols16) - cols15):
#     df_15[c] = pd.NA
# for c in sorted((cols15 | cols16) - cols16):
#     df_16[c] = pd.NA

# concat
df_all = pd.concat([df_15, df_16], ignore_index=True, sort=False)

# ==================================================================
# DÉDUPLICATION
# Pourrait en fait virer toute la déduplication car ici = 0
# On garde si évolution des données ou de choix de clé déduplication
# ==================================================================

# conservation de la clé texte pour éviter de supprimer certaines lignes
# qui ont même uid + id_syceron mais texte différent
# = des didascalies, texte italique, etc.
# si pas de Texte, 370 lignes supprimées

dup_key = ["uid", "id_syceron", "texte"]

# mask pour les lignes qui seraient supprimées par drop_duplicates
mask_removed = df_all.duplicated(subset=dup_key, keep="first")

if mask_removed.sum() == 0:
    print("Pas de doublons avec les clés choisies")
    print(f"concat {len(df_15)} + {len(df_16)} -> {len(df_all)} lignes")
else:
    removed = df_all[mask_removed].copy()
    mask_any = df_all.duplicated(subset=dup_key, keep=False)
    dupe_groups = df_all[mask_any].sort_values(by=dup_key)
    kept_in_groups = df_all[~mask_removed & mask_any]

    print(
        f"Groupes dupliqués distincts : {dupe_groups[dup_key].drop_duplicates().shape[0]}"
    )
    print(f"Lignes supprimées prévues : {len(removed)}")
    display(removed.head())
    display(dupe_groups.head())

    df_all_before = len(df_all)
    df_all = df_all.drop_duplicates(subset=dup_key, keep="first")
    print(
        f"concat {len(df_15)} + {len(df_16)} -> {df_all_before} lignes ; après déduplication {len(df_all)} lignes"
    )

# export
df_all.to_csv("../data/interim/extract_15_16_concat.csv", index=False, encoding="utf-8")
print(f"\n Export CSV : ({df_all.shape[0]} lignes)")

Pas de doublons avec les clés choisies
concat 791311 + 336817 -> 1128128 lignes

 Export CSV : (1128128 lignes)


In [ ]:
# # TODO : deduplication par id_syceron seulement,
# # pour choper les doublons de cas
# # (voir vérif id syceron mais texte différent ?)

# s = df["id_syceron"].dropna()
# dups = s[s.duplicated(keep=False)].unique()

# table_syceron_dups = df.loc[
#     df["id_syceron"].isin(dups),
#     ["uid", "id_acteur", "nom_orateur_clean", "dateSeance_ts", "id_syceron", "texte"],
# ].sort_values(["id_syceron", "dateSeance_ts"])

# display(table_syceron_dups)

# # TODO : virer doublon fichier avec JO
# # CRSJOCGR5L15S2017E1N001 -> CRSANR5L15S2017O1N001 = 2017
# # Et check : CRSJOCGR5L15S2018E1N001 = 2018
# # TODO : investiguer l'autre fichier jo qui ne flague pas, mais genre j'y retrouve pas
# # les id syceron concernés : df[df["id_syceron"]==1360139] check.

# # TODO : aviser pour 16° et CRSCGR5L16S2024O1N001 = congrès.

In [ ]:
# Doublons stricts sur la combinaison id_syceron + texte
tmp_dups = df.loc[
    df["id_syceron"].notna(),
    ["uid", "id_acteur", "nom_orateur_clean", "dateSeance_ts", "id_syceron", "texte"],
].copy()

# Normalisation légère du texte pour éviter les faux écarts d'espaces
tmp_dups["texte_norm"] = (
    tmp_dups["texte"]
    .fillna("")
    .astype(str)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

# Identifier les clés (id_syceron, texte_norm) dupliquées
dups_keys = (
    tmp_dups.groupby(["id_syceron", "texte_norm"], dropna=False)
    .size()
    .reset_index(name="n")
    .query("n > 1")
)

# Table détaillée des lignes concernées
table_syceron_dups = (
    tmp_dups.merge(
        dups_keys[["id_syceron", "texte_norm", "n"]],
        on=["id_syceron", "texte_norm"],
        how="inner",
    )
    .sort_values(["id_syceron", "dateSeance_ts", "uid"])
    .drop(columns=["texte_norm"])
)

print("Nb clés dupliquées (id_syceron + texte) :", len(dups_keys))
print("Nb lignes concernées :", len(table_syceron_dups))
display(table_syceron_dups)